# 10. Validation: Does the Pipeline Work?

1. Example animal: raw behaviour
2. Fit with GS-UM and GS-CP: are they consistent?
3. Fit with SBI: is it consistent? Better?
4. Consensus for this animal
5. All animals: cohort-level accuracy + assignment strip + confusion
6. Misclassified animals: what went wrong?
7. Parameter recovery: GS-UM vs GS-CP vs SBI
8. Calibration: can we trust SBI uncertainty?
9. Summary

In [ ]:
%matplotlib inline
from shared_setup import *
import pickle, torch
from matplotlib.colors import ListedColormap
from matplotlib.patches import Patch
from matplotlib.lines import Line2D

from plotting.validation_report import (
    plot_synth_summary, plot_synth_cv_results,
    plot_synth_model_fits, plot_synth_sbi_diagnostics,
    build_gs_index, extract_gs_recovery, extract_sbi_recovery,
    plot_recovery_overlay, plot_recovery_summary, build_confusion_matrix
)
from behav_utils.plotting.styles import COLOURS

BE_COL, SC_COL = COLOURS['BE'], COLOURS['SC']

In [ ]:
COHORTS = ['static_uniform', 'learning_uniform']

## Load Everything

In [ ]:
# Cohorts
cohort_data = {}
for cohort in COHORTS:
    p = VALIDATION_DIR / 'synthetic_cohorts' / f'{cohort}.pkl'
    if p.exists():
        with open(p, 'rb') as f:
            cohort_data[cohort] = pickle.load(f)['animals']
        print(f'Cohort {cohort}: {len(cohort_data[cohort])} animals')

# GS raw pickles
synth_gs_raw = {}
synth_gs_dfs = {}
for cohort in COHORTS:
    for ft in FIT_TARGETS:
        d = VALIDATION_DIR / 'synth_gs' / f'{cohort}_{ft}'
        if not d.exists(): continue
        sp = d / 'summary.pkl'
        if sp.exists():
            with open(sp, 'rb') as f:
                synth_gs_dfs[(cohort, ft)] = pickle.load(f)['df']
        raws = []
        for p in sorted(d.glob('synth_*.pkl')):
            if p.name == 'summary.pkl': continue
            with open(p, 'rb') as f: raws.append(pickle.load(f))
        if raws: synth_gs_raw[(cohort, ft)] = raws

# SBI pickles
synth_sbi_dfs = {}
for cohort in COHORTS:
    for ft in FIT_TARGETS:
        d = VALIDATION_DIR / 'synth_sbi' / f'{cohort}_{ft}'
        if not d or not d.exists(): continue
        files = sorted(d.glob('synth_*.pkl'))
        if files:
            rows = []
            for p in files:
                with open(p, 'rb') as f: rows.append(pickle.load(f))
            synth_sbi_dfs[(cohort, ft)] = pd.DataFrame(rows)

# SNPE
snpe = load_snpe_networks()

# Build GS index: {fit_target: {animal_id: {'BE': data, 'SC': data}}}
gs_index = build_gs_index(synth_gs_raw)
for ft, animals in gs_index.items():
    print(f'  GS indexed: {ft} — {len(animals)} animals')

# Pick examples with complete GS results for both fit targets
examples = {}
for i, sa in enumerate(cohort_data.get('static_uniform', [])):
    if i == 0:
        continue
    mt = sa['true_model']
    if mt in examples: continue
    aid = sa['animal_id']
    has_both = all(
        ft in gs_index and aid in gs_index[ft]
        and 'BE' in gs_index[ft][aid] and 'SC' in gs_index[ft][aid]
        for ft in FIT_TARGETS
    )
    if has_both:
        examples[mt] = sa
    if len(examples) == 2: break

for mt, sa in examples.items():
    print(f'Example {mt}: {sa["animal_id"]}')

---
## Act 1: What Does This Animal Look Like?

In [ ]:
for mt in ['BE', 'SC']:
    if mt not in examples: continue
    plot_synth_summary(examples[mt])

---
## Act 2: Fitting with GS — UM vs CP

Same animal, two scoring targets. Are the results consistent?

In [ ]:
for mt in ['BE', 'SC']:
    if mt not in examples: continue
    sa = examples[mt]
    print(f'\n{"═" * 50}')
    print(f'  {sa["animal_id"]} (true {mt})')
    print(f'{"═" * 50}')

    for ft in FIT_TARGETS:
        ft_short = FT_LABEL[ft]
        print(f'\n── GS-{ft_short} ──')
        plot_synth_cv_results(sa, gs_index, fit_target=ft)
        plot_synth_model_fits(sa, gs_index, fit_target=ft)

---
## Act 3: SBI — One Coherent Answer

Same posteriors, UM and CP scoring just determine the winner.
Shows: SBI posterior + GS-UM point + GS-CP point + truth.

In [ ]:
from inference.diagnostics import compute_param_stat_correlations
from plotting.sbi import plot_param_stat_correlations

for model in ['be', 'sc']:
    corr_data = compute_param_stat_correlations(
        model_type=model,
        stat_names=snpe[model]['stat_names'],
        burn_in=snpe[model]['burn_in'],
        n_samples=1000,
    )
    plot_param_stat_correlations(
        corr_data,
        title=f'{model.upper()} — Which Stats Inform Which Parameters?',
    )

In [ ]:
for mt in ['BE', 'SC']:
    if mt not in examples: continue
    sa = examples[mt]
    print(f'\n── {sa["animal_id"]} (true {mt}) — SBI ──')
    plot_synth_sbi_diagnostics(
        sa, snpe, gs_index=gs_index, show_ppc=True)

---
## Act 5: All Animals — Cohort-Level Results

Accuracy, assignment strip, confusion matrices.

In [ ]:
# Accuracy table
acc_rows = []
for cohort in COHORTS:
    for ft in FIT_TARGETS:
        row = {'cohort': cohort, 'fit_target': FT_LABEL[ft]}
        key = (cohort, ft)
        for prefix, src in [('gs', synth_gs_dfs), ('sbi', synth_sbi_dfs)]:
            if key in src:
                df = src[key]
                col = 'correct' if 'correct' in df.columns else f'{prefix}_correct'
                if col in df.columns:
                    row[f'{prefix}_acc'] = df[col].mean()
                    row[f'{prefix}_nc'] = int(df[col].sum())
                    row[f'{prefix}_n'] = len(df)
        acc_rows.append(row)
acc_df = pd.DataFrame(acc_rows)
print(acc_df.to_string(index=False, float_format='%.2f'))

In [ ]:
# Accuracy bars
fig, axes = plt.subplots(1, 2, figsize=(12, 5), sharey=True)
ft_colours = {'UM': BE_COL, 'CP': SC_COL}
for ax_i, (method, ac, nc, nn) in enumerate([
    ('Grid Search', 'gs_acc', 'gs_nc', 'gs_n'),
    ('SBI (SNPE)', 'sbi_acc', 'sbi_nc', 'sbi_n'),
]):
    ax = axes[ax_i]
    sub = acc_df.dropna(subset=[ac])
    if len(sub) == 0:
        ax.text(0.5, 0.5, f'No {method} results', transform=ax.transAxes, ha='center')
        ax.set_title(method); continue
    cohorts = sub['cohort'].unique()
    x = np.arange(len(cohorts))
    fts = sub['fit_target'].unique()
    w = 0.35
    for i, ft in enumerate(fts):
        fs = sub[sub['fit_target'] == ft]
        vals, lbls = [], []
        for c in cohorts:
            r = fs[fs['cohort'] == c]
            if len(r): vals.append(r[ac].values[0]); lbls.append(f'{int(r[nc].values[0])}/{int(r[nn].values[0])}')
            else: vals.append(0); lbls.append('')
        bars = ax.bar(x+(-w/2+w*i), vals, w*0.9, label=ft, color=ft_colours.get(ft, f'C{i}'), alpha=0.8)
        for bar, lbl in zip(bars, lbls):
            if lbl: ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.02, lbl, ha='center', va='bottom', fontsize=8)
    ax.set_xticks(x); ax.set_xticklabels([c.replace('_','\n') for c in cohorts], fontsize=9)
    ax.axhline(0.5, color='grey', ls='--', lw=0.8)
    ax.set_ylim(0, 1.15); ax.set_ylabel('Accuracy' if ax_i==0 else '')
    ax.set_title(method, fontsize=12, fontweight='bold'); ax.legend(fontsize=8, loc='lower right')
fig.suptitle('Model Identification Accuracy', fontsize=13, fontweight='bold')
plt.tight_layout(); plt.show()

In [ ]:
# Assignment strip (synthetic — compares to ground truth)
def build_merged_assignments(cohort='static_uniform'):
    if cohort not in cohort_data: return None
    base = pd.DataFrame([{'id': sa['animal_id'], 'true': sa['true_model']} for sa in cohort_data[cohort]])
    for ft in FIT_TARGETS:
        ft_short = FT_LABEL[ft]
        key = (cohort, ft)
        if key in synth_gs_dfs:
            df = synth_gs_dfs[key]
            id_col = 'animal' if 'animal' in df.columns else 'animal_id'
            base = base.merge(df[[id_col,'winner']].rename(columns={id_col:'id','winner':f'GS-{ft_short}'}), on='id', how='left')
        if key in synth_sbi_dfs:
            df = synth_sbi_dfs[key]
            id_col = 'animal_id' if 'animal_id' in df.columns else 'animal'
            base = base.merge(df[[id_col,'winner']].rename(columns={id_col:'id','winner':f'SBI-{ft_short}'}), on='id', how='left')
    return base.sort_values('id').reset_index(drop=True)

strip_df = build_merged_assignments()
if strip_df is not None:
    method_cols = [c for c in strip_df.columns if c not in ('id','true')]
    all_cols = ['true'] + method_cols
    n_m, n_a = len(all_cols), len(strip_df)
    cm = {'BE': 0, 'SC': 1}
    data_mat = np.full((n_a, n_m), np.nan)
    corr_mat = np.ones((n_a, n_m), dtype=bool)
    for i, (_, row) in enumerate(strip_df.iterrows()):
        for j, col in enumerate(all_cols):
            val = row.get(col)
            if isinstance(val, str) and val in cm:
                data_mat[i,j] = cm[val]
                if col != 'true': corr_mat[i,j] = (val == row['true'])
    cmap = ListedColormap([BE_COL, SC_COL])
    fig, ax = plt.subplots(figsize=(1.2*n_m+1, max(6, n_a*0.32)))
    ax.imshow(data_mat, cmap=cmap, vmin=0, vmax=1, aspect='auto')
    for i in range(n_a):
        for j in range(n_m):
            if j==0: continue
            val = strip_df.iloc[i][all_cols[j]]
            if not corr_mat[i, j] and not np.isnan(data_mat[i, j]):
                ax.plot(j, i, 'x', color='white', ms=10, mew=2.5)
            elif np.isnan(data_mat[i, j]):
                fc = '#F0F0F0'
                label = '—' if pd.isna(val) else '?'
                ax.add_patch(plt.Rectangle((j-0.5, i-0.5), 1, 1,
                             facecolor=fc, edgecolor='#CCC', lw=0.5))
                ax.text(j, i, label, ha='center', va='center',
                        fontsize=9, color='#666', fontweight='bold')
    for j in range(n_m+1): ax.axvline(j-0.5, color='white', lw=2)
    ax.axvline(0.5, color='black', lw=2.5)
    ax.set_xticks(range(n_m)); ax.set_xticklabels(['True']+method_cols, fontsize=10, fontweight='bold')
    ax.xaxis.set_ticks_position('top')
    ax.set_yticks(range(n_a)); ax.set_yticklabels(strip_df['id'].values, fontsize=7)
    for j, col in enumerate(all_cols):
        if col=='true': continue
        valid = ~np.isnan(data_mat[:,j])
        if valid.sum()>0: ax.text(j, n_a+0.3, f'{corr_mat[valid,j].mean():.0%}', ha='center', fontsize=9, fontweight='bold')
    ax.legend(handles=[Patch(facecolor=BE_COL,label='BE'),Patch(facecolor=SC_COL,label='SC'),
        Line2D([0],[0],marker='x',color='white',markeredgecolor='white',ms=10,mew=2.5,ls='',label='Wrong')],
        loc='upper left', bbox_to_anchor=(1.02,1), fontsize=9)
    ax.set_title('Model Assignment vs Ground Truth', fontsize=12, fontweight='bold', pad=15)
    plt.tight_layout(); plt.show()
    for col in method_cols:
        valid = strip_df[col].isin(['BE','SC'])
        if valid.sum()>0:
            acc = (strip_df.loc[valid,col]==strip_df.loc[valid,'true']).mean()
            print(f'  {col:10s}: {acc:.0%}')

In [ ]:
# Confusion matrices
cms = {}
for cohort in COHORTS:
    for ft in FIT_TARGETS:
        lc = cohort.replace('_',' ').title()
        key = (cohort,ft)
        if key in synth_gs_dfs:
            cm = build_confusion_matrix(synth_gs_dfs[key], true_col='true_model', pred_col='winner')

            if cm is not None: cms[f'GS-{FT_LABEL[ft]}\n{lc}'] = cm
        if key in synth_sbi_dfs:
            cm = build_confusion_matrix(synth_sbi_dfs[key], true_col='true_model', pred_col='winner')
            if cm is not None: cms[f'SBI-{FT_LABEL[ft]}\n{lc}'] = cm

if cms:
    n = len(cms)
    fig, axes = plt.subplots(1,n,figsize=(3.2*n,3.8)); axes = np.atleast_1d(axes)
    for ax,(title,cm) in zip(axes,cms.items()):
        cm_n = cm/cm.sum(axis=1,keepdims=True)
        ax.imshow(cm_n,cmap='Blues',vmin=0,vmax=1,aspect='equal')
        for i in range(2):
            for j in range(2):
                c = 'white' if cm_n[i,j]>0.5 else 'black'
                ax.text(j,i,f'{cm[i,j]}\n({cm_n[i,j]*100:.0f}%)',ha='center',va='center',fontsize=12,fontweight='bold',color=c)
        ax.set_xticks([0,1]);ax.set_xticklabels(['BE','SC'])
        ax.set_yticks([0,1]);ax.set_yticklabels(['BE','SC'])
        ax.set_xlabel('Assigned');ax.set_ylabel('True')
        tot=cm.sum(); ax.set_title(f'{title}\n{cm.trace()}/{tot} ({cm.trace()/tot:.0%})',fontsize=9,fontweight='bold')
        for s in ax.spines.values(): s.set_visible(False)
    fig.suptitle('Confusion Matrices',fontsize=13,fontweight='bold')
    plt.tight_layout(); plt.show()

---
## Act 6: Misclassified Animals

Drill into animals where GS got it wrong.

In [ ]:
rk = ('static_uniform', 'update_matrix')
if rk in synth_gs_dfs:
    df = synth_gs_dfs[rk]
    cc = 'correct' if 'correct' in df.columns else 'gs_correct'
    if cc in df.columns:
        wrong = df[~df[cc]]
        aid_col = 'animal' if 'animal' in wrong.columns else 'animal_id'
        print(f'{len(wrong)} misclassified by GS-UM:')
        for _, r in wrong.iterrows():
            print(f'  {r[aid_col]}: true={r["true_model"]}, assigned={r["winner"]}')

        for _, r in wrong.head(5).iterrows():
            aid = r[aid_col]
            sa = next((a for a in cohort_data['static_uniform']
                       if a['animal_id'] == aid), None)
            if sa:
                print(f'\n── {aid} (true: {sa["true_model"]}, assigned: {r["winner"]}) ──')
                plot_synth_summary(sa)
                plot_synth_cv_results(sa, gs_index, fit_target='update_matrix')
                plot_synth_model_fits(sa, gs_index, fit_target='update_matrix')

---
## Act 7: Parameter Recovery — GS-UM vs GS-CP vs SBI

Three recovery tracks on the same axes.

In [ ]:
gs_um_recovery = extract_gs_recovery(
    synth_gs_raw.get(('static_uniform', 'update_matrix'), []))
gs_cp_recovery = extract_gs_recovery(
    synth_gs_raw.get(('static_uniform', 'conditional_psych'), []))

if 'static_uniform' in cohort_data and len(snpe) == 2:
    print('Re-conditioning SNPE on all synthetic animals...')
    sbi_recovery = extract_sbi_recovery(cohort_data['static_uniform'], snpe)
    for mt in ['BE', 'SC']:
        if sbi_recovery[mt]:
            n = len(next(iter(sbi_recovery[mt].values()))['true'])
            print(f'  SBI {mt}: {n} animals')
else:
    sbi_recovery = {'BE': {}, 'SC': {}}

for label, rec in [('GS-UM', gs_um_recovery), ('GS-CP', gs_cp_recovery)]:
    for mt in ['BE', 'SC']:
        if rec.get(mt):
            n = len(next(iter(rec[mt].values()))['true'])
            print(f'{label} {mt}: {n} animals')

In [ ]:
plot_recovery_overlay(gs_um_recovery, gs_cp_recovery, sbi_recovery)

In [ ]:
plot_recovery_summary(gs_um_recovery, gs_cp_recovery, sbi_recovery)

In [ ]:
# "What SBI Adds" — one example: posterior + GS-UM + GS-CP + truth
if examples.get('BE'):
    sa = examples['BE']
    print(f'Example: {sa["animal_id"]} — GS gives two numbers, SBI gives one distribution')
    plot_synth_sbi_diagnostics(sa, snpe, gs_index=gs_index, show_ppc=False)

---
## Act 8: Calibration

Do X% CIs contain the true value X% of the time?

In [ ]:
if 'static_uniform' in cohort_data and len(snpe) == 2:
    ci_levels = [10, 20, 30, 40, 50, 60, 70, 80, 90, 95]
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    for ax_i, mt in enumerate(['BE', 'SC']):
        ax = axes[ax_i]
        animals = [a for a in cohort_data['static_uniform'] if a['true_model'] == mt]
        if not animals: continue
        mk = mt.lower()
        pnames = snpe[mk]['param_names']
        stat_names = snpe[mk]['stat_names']
        all_samples, all_true = {}, {}
        for ai, sa in enumerate(animals):
            td = {k: v for k, v in (vars(sa['true_params']) if hasattr(sa['true_params'],'__dict__') else sa['true_params']).items()
                  if not str(k).startswith('_') and isinstance(v, (int, float))}
            try:
                all_stats = []
                for sess in sa['sessions']:
                    s = compute_summary_stats(
                        sess.trials.choice, sess.trials.stimulus, sess.trials.category,
                        stat_names=stat_names, return_dict=False)
                    all_stats.append(s)
                x_obs = np.nanmean(all_stats, axis=0)
                if np.any(np.isnan(x_obs)): continue
                samples = snpe[mk]['posterior'].sample(
                    (5000,), x=torch.tensor(x_obs, dtype=torch.float32)).numpy()
                all_samples[ai] = samples; all_true[ai] = td
            except Exception: continue
        for pj, pn in enumerate(pnames):
            coverage = []
            for ci in ci_levels:
                lo_p, hi_p = (100-ci)/2, 100-(100-ci)/2
                hits, total = 0, 0
                for ai in all_samples:
                    if pn not in all_true[ai]: continue
                    lo = np.percentile(all_samples[ai][:,pj], lo_p)
                    hi = np.percentile(all_samples[ai][:,pj], hi_p)
                    hits += (lo <= all_true[ai][pn] <= hi); total += 1
                coverage.append(hits/total if total>0 else np.nan)
            ax.plot(np.array(ci_levels)/100, coverage, 'o-', label=pn, ms=4)
        ax.plot([0,1],[0,1],'k--',alpha=0.3,label='Perfect')
        ax.set(xlabel='CI level',ylabel='Fraction in CI',xlim=(0,1),ylim=(0,1),aspect='equal')
        ax.legend(fontsize=8)
        ax.set_title(f'{mt} (n={len(all_samples)})',fontsize=11,fontweight='bold')
    fig.suptitle('SBI Calibration',fontsize=13,fontweight='bold')
    plt.tight_layout(); plt.show()

---
## Act 9: Summary + Drill-Down

In [ ]:
print('═══ KEY NUMBERS ═══\n')
print('Model identification:')
for _, r in acc_df.iterrows():
    line = f'  {r["cohort"]} / {r["fit_target"]}:'
    if pd.notna(r.get('gs_acc')): line += f'  GS={r["gs_acc"]:.0%} ({r["gs_nc"]:.0f}/{r["gs_n"]:.0f})'
    if pd.notna(r.get('sbi_acc')): line += f'  SBI={r["sbi_acc"]:.0%} ({r["sbi_nc"]:.0f}/{r["sbi_n"]:.0f})'
    print(line)
print('\nParameter recovery:')
for label, rec in [('GS-UM', gs_um_recovery), ('GS-CP', gs_cp_recovery), ('SBI', sbi_recovery)]:
    for mt in ['BE', 'SC']:
        for pn, vals in rec.get(mt, {}).items():
            t, r = np.array(vals['true']), np.array(vals['recovered'])
            if len(t)>2: print(f'  {label:6s} {mt} {pn:20s} r={np.corrcoef(t,r)[0,1]:.3f}')

In [ ]:
# Drill-down
INSPECT_ID = 'BE_static_03'  # ← change
sa = next((a for a in cohort_data.get('static_uniform', [])
           if a['animal_id'] == INSPECT_ID), None)
if sa is None:
    print(f'{INSPECT_ID} not found. Available:')
    for a in cohort_data.get('static_uniform', []):
        print(f'  {a["animal_id"]} ({a["true_model"]})')
else:
    print(f'Inspecting {INSPECT_ID} (true {sa["true_model"]})')
    plot_synth_summary(sa)
    for ft in FIT_TARGETS:
        plot_synth_cv_results(sa, gs_index, fit_target=ft)
    plot_synth_sbi_diagnostics(sa, snpe, gs_index=gs_index)